In [0]:
import pyspark.sql.functions as F

catalog_name = "f1_dev"
target_table = f"{catalog_name}.silver.dim_ferrari_stock"



In [0]:
# 1. Read Raw Stock Data
df_raw = spark.table(f"{catalog_name}.bronze.ferrari_stock_raw")



In [0]:
print(df_raw.printSchema())
print(df_raw.show(5))


In [0]:
# 2. Transform & Cleanse
df_stock_silver = (
    df_raw
    .replace(r"\N", None)
    # Convert Date format (handles MM/dd/yyyy or yyyy-MM-dd)
    .withColumn("trade_date", F.to_date(F.col("Date"), "MM/dd/yyyy"))
    # Extract readable day of week (e.g., 'Monday', 'Tuesday')
    .withColumn("day_of_week", F.date_format(F.col("trade_date"), "EEEE"))
    # Strip '$' and ',' characters and cast numeric columns
    .withColumn("close_price", F.regexp_replace(F.col("Close/Last"), r"[\$,]", "").cast("double"))
    .withColumn("open_price", F.regexp_replace(F.col("Open"), r"[\$,]", "").cast("double"))
    .withColumn("high_price", F.regexp_replace(F.col("High"), r"[\$,]", "").cast("double"))
    .withColumn("low_price", F.regexp_replace(F.col("Low"), r"[\$,]", "").cast("double"))
    .withColumn("volume", F.col("Volume").cast("long"))
    .select("trade_date", "day_of_week", "close_price", "open_price", "high_price", "low_price", "volume")
    .filter(F.col("trade_date").isNotNull())
)



In [0]:
# 3. Write to Silver
df_stock_silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table)



In [0]:
# 4. Governance Tags
spark.sql(f"""
    ALTER TABLE {target_table} SET TAGS (
        'layer' = 'silver',
        'table_type' = 'dimension',
        'domain' = 'financial'
    )
""")

print(f"✓ Successfully built and governed: {target_table}")